# **Smart Water Systems: K-Means Clustering**

## Learning goals

By the end of this notebook, students should be able to:

1. Explain the purpose of **clustering**.
2. Represent daily water-use profiles as numeric vectors.
3. Apply **k-means clustering** to group similar daily profiles.
4. Interpret cluster labels and cluster centers.
5. Use the **sum of squared errors (SSE)** to help select the number of clusters.

We will use the same ECJ water-use data from the query-by-content lecture.

# **1. Load the data**

In [ ]:
# import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

# Define course folders
course_folder = '/content/drive/MyDrive/CE374U2_Fall2026'
data_folder = os.path.join(course_folder, 'data')

file_name = '03_ECJ.csv'
data_file = os.path.join(data_folder, file_name)

data_file

In [ ]:

df = pd.read_csv(data_file)
df.head()

### Prepare the time variable

Convert the timestamp to a datetime variable and keep the water-use data.

In [ ]:
column_name = 💡  # Change to 'Elec' for electricity data

df['DateTime'] = pd.to_datetime(df['DateTime'])
df = df[['DateTime', column_name]]
df.head()

In [ ]:
print('Start:', df['DateTime'].min())
print('End:', df['DateTime'].max())
print('Number of records:', len(df))
print('Missing values:', df[column_name].isna().sum())

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(df['DateTime'], df[column_name])
plt.xlabel('Time')
plt.ylabel('Demand (Gallons)')
plt.title('Training and test periods')
plt.show()

In [ ]:
import plotly.express as px
from google.colab import files

fig = px.line(
    df,
    x='DateTime',
    y=column_name,
    title='Water Use and Closest Matching Profiles'
)
fig.show()

# html_file_name = file_name.rsplit('.', 1)[0] + '_html.html'
# fig.write_html(html_file_name)
# files.download(html_file_name)

# **2. Reshape the time series into daily profiles**

As in the previous lecture, we want each day to be one object that can be compared with other days.

We first create:

- `date` = calendar day
- `hour` = hour of the day

Then we reshape the hourly time series so that:

- each **row** represents one day
- each **column** represents one hour
- each row contains a 24-hour water-use profile

In [ ]:
df['date'] = df['DateTime'].dt.date
df['hour'] = df['DateTime'].dt.hour

df.head()

In [ ]:
daily = df.pivot(index='date', columns='hour', values=column_name)

daily.head()

A complete daily profile should contain 24 hourly values. We remove days with missing hourly values before clustering.

In [ ]:
print('Number of days before removing incomplete days:', len(daily))

daily = daily.dropna()

print('Number of complete days:', len(daily))

### Convert the daily profiles to a NumPy array

`daily` is a **pandas DataFrame**. Pandas is useful for organizing the data because the dates and hours remain attached to the water-use values.

For the clustering calculations, we use a **NumPy array** containing only the numeric profiles.

If there are $n$ days, the array has the form:

$$
X =
\begin{bmatrix}
x_{1,0} & x_{1,1} & \cdots & x_{1,23} \\
x_{2,0} & x_{2,1} & \cdots & x_{2,23} \\
\vdots & \vdots & & \vdots \\
x_{n,0} & x_{n,1} & \cdots & x_{n,23}
\end{bmatrix}
$$

Each **row** is one day and each **column** is one hour.

In [ ]:
profiles = daily.to_numpy(dtype=float)

profiles.shape

# **3. What is clustering?**

In query-by-content, we selected one profile and searched for other profiles that were similar to it.

In **clustering**, we do not select a query. Instead, we ask the algorithm to find groups of similar profiles in the data.

K-means requires us to choose the number of clusters, $k$.

The algorithm then:

1. starts with $k$ cluster centers
2. assigns each daily profile to the nearest center
3. updates each center using the profiles assigned to it
4. repeats until the assignments stop changing

We will start by manually choosing:

$$
k = 3
$$

# **4. Apply k-means clustering**

### K-means using scikit-learn

We use `KMeans` from the **scikit-learn** Python library to group daily profiles with similar shapes. Each row in `profiles` represents one day, and each column represents a time step.

The command `KMeans(n_clusters=k, random_state=0, n_init=10)` sets up the clustering model:

- **`n_clusters=k`** specifies the number of clusters. We begin with `k = 3`.
- **`random_state=0`** makes the initialization reproducible, so running the same code on the same data gives the same result. The value `0` is a seed, not a cluster number.
- **`n_init=10`** runs k-means with 10 different sets of initial centroids and keeps the result with the lowest sum of squared errors (SSE). These are 10 separate runs, not 10 iterations.

The command `kmeans.fit(profiles)` applies the algorithm to the data. Within each run, it repeatedly assigns each daily profile to its nearest centroid using Euclidean distance and updates each centroid to the mean of its assigned profiles, until the stopping criterion is met.

In [ ]:
# import kmeans package
from sklearn.cluster import KMeans

In [ ]:
k = 💡

kmeans = KMeans(n_clusters=💡, random_state=0, n_init=10)

The `fit()` command applies the k-means algorithm to the daily profiles.

In [ ]:
kmeans.fit(💡)

After fitting the model, `labels_` contains the cluster assigned to each daily profile.

Because we selected three clusters, each label will be 0, 1, or 2.

In [ ]:
labels = kmeans.💡

labels[:20]

The number of labels should equal the number of daily profiles.

In [ ]:
print('Number of daily profiles:', len(profiles))
print('Number of cluster labels:', len(labels))

### **How many days are in each cluster?**


After applying K-means, each day is assigned to a cluster. We want to count how many days were assigned to each cluster.

The code uses three Python operations:

1. **`np.unique(labels, return_counts=True)`** finds the unique cluster labels and counts how many times each appears.
   - `unique_labels` stores the cluster numbers.
   - `counts` stores the number of days in each cluster.

2. **`zip(unique_labels, counts)`** pairs each cluster label with its corresponding count. For example:
   - `unique_labels = [0, 1, 2]`
   - `counts = [10, 8, 12]`
   - `zip()` pairs them as `(0, 10)`, `(1, 8)`, `(2, 12)`.

3. **`for label, count in ...`** loops through these pairs. In each iteration, `label` and `count` take the values from one pair, and the `print()` statement displays the result.

**Question:** Are the days distributed evenly across the clusters, or are some clusters more common than others?

In [ ]:
# Find unique cluster labels and count their occurrences
unique_labels, counts = np.unique(labels, return_counts=True)

# Loop through each cluster and its corresponding count
for label, count in zip(unique_labels, counts):
    print('Cluster', label, ':', count, 'days')

# **5. Examine the cluster centers**

Each cluster has a **center**, also called a centroid.

For our daily profiles, each center contains 24 values:

$$
C_j = [C_{j,0}, C_{j,1}, \ldots, C_{j,23}]
$$

The center represents the average 24-hour pattern of the profiles assigned to that cluster.

In [ ]:
centers = kmeans.💡

centers.💡

Plot the cluster centers to compare their typical daily water-use patterns.

In [ ]:
plt.figure(figsize=(10, 5))

for i in range(k):
    plt.plot(range(24), centers[i], marker='o', label='Cluster ' + str(i))

plt.xlabel('Hour')
plt.ylabel('Utility use')
plt.title('Cluster Centers')
plt.xticks(range(0, 24, 3))
plt.legend()
plt.grid(True)

plt.show()

### View the profiles within each cluster

The centroid shows the typical pattern for a cluster, but it is also useful to see the individual daily profiles assigned to that cluster.

For each cluster:

- the thin lines show the individual daily profiles assigned to the cluster
- the thick black line shows the cluster centroid

This lets us see both the typical pattern and the variation within each cluster.

In [ ]:
cluster = 💡

cluster_profiles = profiles[labels == 💡]

plt.figure(figsize=(9, 5))

for profile in cluster_profiles:
    plt.plot(range(24), profile, linewidth=1, alpha=0.25)

plt.plot(
    range(24),
    centers[cluster],
    color='black',
    linewidth=4,
    label='Centroid'
)

plt.xlabel('Hour')
plt.ylabel('Water use')
plt.title('Cluster 0')
plt.xticks(range(0, 24, 3))
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
cluster = 💡

cluster_profiles = profiles[labels == 💡]

plt.figure(figsize=(9, 5))

for profile in cluster_profiles:
    plt.plot(range(24), profile, linewidth=1, alpha=0.25)

plt.plot(
    range(24),
    centers[cluster],
    color='black',
    linewidth=4,
    label='Centroid'
)

plt.xlabel('Hour')
plt.ylabel('Water use')
plt.title('Cluster 1')
plt.xticks(range(0, 24, 3))
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
cluster = 2

cluster_profiles = profiles[labels == cluster]

plt.figure(figsize=(9, 5))

for profile in cluster_profiles:
    plt.plot(range(24), profile, linewidth=1, alpha=0.25)

plt.plot(
    range(24),
    centers[cluster],
    color='black',
    linewidth=4,
    label='Centroid'
)

plt.xlabel('Hour')
plt.ylabel('Water use')
plt.title('Cluster 2')
plt.xticks(range(0, 24, 3))
plt.legend()
plt.grid(True)

plt.show()

These plots help us interpret the clusters.

A centroid summarizes the average pattern in a cluster, while the individual profiles show how much variation exists around that centroid.

Consider:

1. Are the profiles within each cluster tightly grouped or widely spread?
2. Do the clusters differ mainly in magnitude, shape, or both?
3. Does the centroid appear to be a good representation of the profiles assigned to the cluster?

### Add the cluster labels back to the dates

The clustering calculations were performed using the NumPy array `profiles`.

We can now connect the cluster labels back to the dates stored in the pandas DataFrame.

In [ ]:
cluster_results = pd.DataFrame({
    'date': daily.index,
    'cluster': labels
})

cluster_results.head(10)

We can also look at some dates assigned to each cluster.

In [ ]:
for i in range(k):
    print('\nCluster', i)
    print(cluster_results[cluster_results['cluster'] == i].head())

# **6. Choosing the number of clusters**

So far, we selected $k=3$ manually.

But how do we decide whether the data should contain 2, 3, 4, or more clusters?

One simple approach is to compare the **sum of squared errors (SSE)** for different values of $k$.

For each profile, k-means calculates its squared distance from the center of its assigned cluster. SSE adds these squared distances across all profiles.

$$
SSE = \sum_{i=1}^{n} \left\| X_i - C_{z_i} \right\|^2
$$

where:

- $X_i$ is daily profile $i$
- $C_{z_i}$ is the center of the cluster assigned to profile $i$

A smaller SSE means the profiles are closer to their assigned cluster centers.

### Calculate SSE for different values of $k$

We will test several possible numbers of clusters and store the SSE from each model.

In scikit-learn, the SSE is stored as `inertia_`.

In [ ]:
k_values = range(1, 9)

sse = []

For each value of $k$:

1. create a k-means model
2. fit the model to the daily profiles
3. save the SSE

In [ ]:
for k in k_values:
    model = KMeans(
        n_clusters=💡,
        random_state=0,
        n_init=10
    )

    model.fit(💡)

    sse.append(model.💡)

In [ ]:
sse

### Plot the SSE

SSE will always decrease as the number of clusters increases.

We look for a point where adding more clusters produces only a small additional reduction in SSE. This bend in the curve is often called the **elbow** or **knee**.

The elbow can help us choose a reasonable number of clusters, but the final choice still requires interpretation.

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(k_values, sse, marker='o')

plt.xlabel('Number of clusters, k')
plt.ylabel('SSE')
plt.title('SSE for Different Numbers of Clusters')
plt.xticks(k_values)
plt.grid(True)

plt.show()

### Select the final number of clusters

Examine the SSE plot and choose a value of $k$ where the curve begins to flatten.

Enter your selected value below.

In [ ]:
final_k = 💡

# **7. Fit the final clustering model**

Now repeat k-means using the selected number of clusters.

In [ ]:
final_model = KMeans(
    n_clusters=💡,
    random_state=0,
    n_init=10
)

final_model.fit(profiles)

In [ ]:
final_labels = final_model.labels_
final_centers = final_model.cluster_centers_

Plot the final cluster centers.

In [ ]:
plt.figure(figsize=(10, 5))

for i in range(final_k):
    plt.plot(
        range(24),
        final_centers[i],
        marker='o',
        label='Cluster ' + str(i)
    )

plt.xlabel('Hour')
plt.ylabel('Water use')
plt.title('Final Cluster Centers')
plt.xticks(range(0, 24, 3))
plt.legend()
plt.grid(True)

plt.show()

# **8. Interpretation**

Consider the final cluster centers.

1. How do the daily patterns differ among the clusters?
2. Do the clusters mainly differ in **magnitude**, **shape**, or both?
3. What type of day might each cluster represent?
4. Are the cluster sizes similar, or are some daily patterns much more common than others?
5. How is clustering different from the query-by-content analysis in the previous lecture?

## **Takeaway**

K-means clustering groups similar daily profiles without requiring a predefined query or class label.

In this notebook:

- each day was represented as a 24-hour vector
- k-means grouped similar vectors
- cluster centers summarized the typical pattern in each group
- SSE helped us examine possible values of $k$

The same daily-profile matrix can also be used with other clustering methods and other distance measures.

# **9. In-class exercise: K-means clustering of utility demand**

**Goal:** Explore how outliers and the choice of $k$ affect clustering, then compare electricity and water-use profiles. Run the provided code and answer the questions in the accompanying worksheet.

### Part A. Explore raw electricity profiles
1. Set `column_name = 'Elec'`. Plot the raw time series and identify any unusual observations.
2. Set `k = 3`, fit k-means, and examine the cluster centers and the number of days in each cluster. Does any cluster appear to represent an outlier rather than a typical day?
3. Change to `k = 5` and rerun the clustering cells. How do the cluster centers and cluster sizes change? Does increasing $k$ solve the outlier problem?

### Part B. Cap high outliers
4. Return to the preprocessing cell. Calculate $Q_1$, $Q_3$, and $IQR = Q_3-Q_1$, then cap values above $Q_3+1.5IQR$ using the provided code. **Reload the original CSV first**, so you can compare the raw and capped data.
5. Recreate the daily profiles and repeat k-means with `k = 3`. Compare the results with Part A. Are the cluster centers more representative of typical days? Explain.

### Part C. Choose $k$ using SSE
6. Run the SSE loop for $k = 1,2,\ldots,8$ using the capped electricity data. Plot SSE versus $k$.
7. Choose a final $k$. Is there a clear elbow? Support your choice using both the SSE curve and the cluster-center plots. Remember that SSE decreases as $k$ increases, so the lowest SSE alone is not the selection criterion.

### Part D. Repeat for water
8. Reload the original CSV and set `column_name = 'Water'` (verify the exact column name in your CSV). Apply the **same IQR rule**, calculating its threshold from the water data.
9. Repeat the daily-profile clustering and SSE analysis. Choose $k$ and examine the final cluster centers.
10. **Interpretation:** Are the clusters identifying different *shapes* of daily use, or mainly separating days by *how much* utility use occurs? What feature of the current approach might explain your answer?

**Submission:** Record your answers in the worksheet. Include the requested cluster-center and SSE plots.


## **Handling Outliers Using the Interquartile Range (IQR)**
We will identify unusually high values and limit their effect on the analysis.

The code uses the following steps:

1. **`quantile()`** calculates percentiles of the data:
   - `Q1` is the 25th percentile (25% of values are at or below Q1).
   - `Q3` is the 75th percentile (75% of values are at or below Q3).

2. **`IQR = Q3 - Q1`** calculates the interquartile range, which measures the spread of the middle 50% of the data.

3. **`threshold = Q3 + 1.5 * IQR`** defines an upper threshold. Values above this threshold are considered unusually high.

4. **`clip(upper=threshold)`** replaces any value above the threshold with the threshold itself. Values below the threshold remain unchanged.

5. **`df[column_name] = ...`** saves the modified values back into the DataFrame.

**Note:** We are not removing outliers or deleting rows. We are limiting extreme values to reduce their influence on the clustering results.

**Question:** Why might extreme values affect K-means clustering?

In [ ]:
Q1 = df[column_name].quantile(0.25)
Q3 = df[column_name].quantile(0.75)
IQR = Q3 - Q1

threshold = Q3 + 1.5 * IQR
print(threshold)
df[column_name] = df[column_name].clip(upper=threshold)